In [ ]:
import scanpy as sc
import seaborn as sns
import harmonypy as hm
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import palantir
import pandas as pd

In [ ]:
adata = sc.read("../results/adata/09-annotation.h5ad")

In [ ]:
adata = adata[adata.obs["cell_type"] == "muscle"].copy()

In [ ]:
adata.X = adata.layers["soupx_rounded"]

In [ ]:
adata.obs["sample"].value_counts()

In [ ]:
print(f"{len(adata.var)} genes before filtering")
sc.pp.filter_genes(adata, min_counts=10)
print(f"{len(adata.var)} genes after filtering")

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
sns.histplot(adata.X.sum(1), bins=100)

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000)

In [ ]:
sc.pl.highly_variable_genes(adata, log=True)

In [ ]:
sc.pp.pca(adata, svd_solver="arpack", mask_var="highly_variable")

In [ ]:
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

In [ ]:
sc.pl.pca_scatter(adata, color="sample", components=["1,2", "3,4"])

In [ ]:
harmony_out = hm.run_harmony(adata.obsm["X_pca"], adata.obs, "sample", max_iter_harmony=30)
adata.obsm['X_pca_harmony'] = harmony_out.Z_corr

In [ ]:
sc.pl.embedding(adata, basis="X_pca_harmony", color=["sample", "day"])

In [ ]:
# sc.pp.neighbors(adata, n_pcs=40)
sc.pp.neighbors(adata, n_pcs=40, use_rep='X_pca_harmony')
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, ncols=3, color=["sample", "day"])

In [ ]:
sc.tl.leiden(adata, key_added="cluster", resolution=0.15, flavor="igraph")

In [ ]:
sc.pl.umap(adata, color="cluster", legend_loc="on data")

In [ ]:
# marker_genes = {
#     "MuSCs & progenitors": ["Pax7"],
#     "Mature skeletal muscle": ["Myod1", "Acta1"],
# }
# for cell_type, genes in marker_genes.items():
#     print(f"{cell_type}:")
#     sc.pl.umap(adata, color=["cluster"] + genes, title=["Clusters"] + genes)

In [ ]:
sc.pl.umap(adata, color="day", mask_obs=(adata.obs["day"] == 0))

In [ ]:
sc.pl.umap(adata, color="day", mask_obs=(adata.obs["day"] == 2))

In [ ]:
sc.pl.umap(adata, color="day", mask_obs=(adata.obs["day"] == 5))

In [ ]:
sc.pl.umap(adata, color="day", mask_obs=(adata.obs["day"] == 7))

In [ ]:
sc.tl.score_genes(
    adata,
    gene_list=["Pax7", "Foxo3", "Vcam1"],
    score_name="quiescence_score"
)

In [ ]:
sc.pl.umap(adata, color="quiescence_score")

In [ ]:
adata.uns["iroot"] = root_ix = np.argmax(adata.obs["quiescence_score"])

In [ ]:
sc.tl.diffmap(adata)

In [ ]:
sc.pl.diffmap(
    adata,
    color="cluster",
    components=["1,2", "2,3", "3,4"])

In [ ]:
# Index for diffusion component is component-1
diff_comp1 = 1
diff_comp2 = 2
f, ax = plt.subplots()
sns.scatterplot(x=adata.obsm["X_diffmap"][:,diff_comp1], 
                y=adata.obsm["X_diffmap"][:,diff_comp2],  
                hue=adata.obs["cluster"], ax=ax)

sns.scatterplot(x=[adata.obsm["X_diffmap"][root_ix, diff_comp1]],
                y=[adata.obsm["X_diffmap"][root_ix,diff_comp2]],
                marker="*", c="black", s=150, ax=ax)

In [ ]:
f, ax = plt.subplots()
sc.pl.umap(adata, color="day", ax=ax, show=False)
# sc.pl.embedding(adata, basis="X_pca_harmony", color="day", ax=ax, show=False)
sns.scatterplot(
    x=[
        adata.obsm["X_umap"][root_ix][0],
      ],
    y=[
        adata.obsm["X_umap"][root_ix][1],
      ],
    marker="*", c="black", s=150, ax=ax)
plt.show()

In [ ]:
# Index for diffusion component is component-1
root_ix = adata.obsm["X_diffmap"][:,1].argmax()
adata.uns["iroot"] = root_ix

term1 = adata.obsm["X_diffmap"][:,2].argmax()
term2 = adata.obsm["X_diffmap"][:,2].argmin()

In [ ]:
# Index for diffusion component is component-1
diff_comp1 = 1
diff_comp2 = 2
f, ax = plt.subplots()
sns.scatterplot(x=adata.obsm["X_diffmap"][:,diff_comp1], 
                y=adata.obsm["X_diffmap"][:,diff_comp2],  
                hue=adata.obs["cluster"], ax=ax)

sns.scatterplot(x=[adata.obsm["X_diffmap"][root_ix, diff_comp1]],
                y=[adata.obsm["X_diffmap"][root_ix, diff_comp2]],
                marker="*", c="black", s=150, ax=ax)

sns.scatterplot(x=[adata.obsm["X_diffmap"][term1, diff_comp1]],
                y=[adata.obsm["X_diffmap"][term1, diff_comp2]],
                marker="*", c="black", s=150, ax=ax)

sns.scatterplot(x=[adata.obsm["X_diffmap"][term2, diff_comp1]],
                y=[adata.obsm["X_diffmap"][term2, diff_comp2]],
                marker="*", c="black", s=150, ax=ax)


In [ ]:
f, ax = plt.subplots()
sc.pl.umap(adata, color="day", ax=ax, show=False)
# sc.pl.embedding(adata, basis="X_pca_harmony", color="day", ax=ax, show=False)
sns.scatterplot(
    x=[
        adata.obsm["X_umap"][root_ix][0],
        adata.obsm["X_umap"][term1][0],
        adata.obsm["X_umap"][term2][0],
      ],
    y=[
        adata.obsm["X_umap"][root_ix][1],
        adata.obsm["X_umap"][term1][1],
        adata.obsm["X_umap"][term2][1],
      ],
    marker="*", c="black", s=150, ax=ax)
plt.show()

In [ ]:
sc.tl.dpt(adata)

In [ ]:
sc.pl.umap(adata, color="dpt_pseudotime")

In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d

N = 100 # Number of genes
n_bins = 50 # Number of bins
smooth_window = 1

# Select top N highly variable genes by normalized dispersion
top_genes = (
    adata.var[adata.var["highly_variable"]]
    .nlargest(N, "dispersions_norm")
    .index.tolist()
)

# Extract expression matrix
X = adata[:, top_genes].X
if hasattr(X, "toarray"):
    X = X.toarray()
X = np.array(X, dtype=float)

# Sort cells by pseudotime (drop NaN cells like the root)
pt = adata.obs["dpt_pseudotime"].values
valid = np.isfinite(pt)
X, pt = X[valid], pt[valid]
order = np.argsort(pt)
X_sorted, pt_sorted = X[order], pt[order]

# Bin cells along pseudotime and average
bins = np.linspace(pt_sorted.min(), pt_sorted.max(), n_bins + 1)
bin_idx = np.clip(np.digitize(pt_sorted, bins) - 1, 0, n_bins - 1)
X_binned = np.array([X_sorted[bin_idx == b].mean(axis=0) if (bin_idx == b).any()
                     else np.zeros(len(top_genes)) for b in range(n_bins)])

# Smooth and z-score each gene
X_binned = uniform_filter1d(X_binned, size=smooth_window, axis=0)
std = X_binned.std(axis=0)
std[std == 0] = 1
X_z = (X_binned - X_binned.mean(axis=0)) / std

# Order genes by peak expression bin
gene_order = np.argsort(np.argmax(X_z, axis=0))
X_z = X_z[:, gene_order]
genes_ordered = [top_genes[i] for i in gene_order]

# Plot
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(X_z.T, aspect="auto", cmap="RdYlBu_r", vmin=-2, vmax=2)
ax.set_yticks(range(len(genes_ordered)))
ax.set_yticklabels(genes_ordered, fontsize=7)
ax.set_xlabel("Pseudotime →")
ax.set_title(f"Top {N} HVGs Along Pseudotime")
xtick_pos = np.linspace(0, n_bins - 1, 6, dtype=int)
ax.set_xticks(xtick_pos)
ax.set_xticklabels([f"{pt_sorted.min() + (pt_sorted.max() - pt_sorted.min()) * i / 5:.2f}"
                    for i in range(6)], fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.6, label="Z-score")
plt.tight_layout()
plt.show()

## Palantir

In [ ]:
dm_res = palantir.utils.run_diffusion_maps(adata, n_components=5)

In [ ]:
ms_data = palantir.utils.determine_multiscale_space(adata)

In [ ]:
imputed_X = palantir.utils.run_magic_imputation(adata)

In [ ]:
palantir.plot.plot_diffusion_components(adata)
plt.show()



In [ ]:
start_cell = adata.obs_names[root_ix]
pr_res = palantir.core.run_palantir(adata, start_cell, num_waypoints=500)

In [ ]:
palantir.plot.plot_palantir_results(adata, s=3)
plt.show()

In [ ]:
terminal_cell1 = adata.obs_names[term1]
terminal_cell2 = adata.obs_names[term2]

terminal_states = {
    "State1": adata.obs_names[term1], 
    "State2": adata.obs_names[term2]
}

pr_res = palantir.core.run_palantir(adata, start_cell, num_waypoints=500, terminal_states=[terminal_cell1, terminal_cell2])

In [ ]:
palantir.plot.plot_palantir_results(adata, s=3)
plt.show()

In [ ]:
masks = palantir.presults.select_branch_cells(adata, q=.01, eps=.01)

In [ ]:
masks[:,0]

In [ ]:
palantir.plot.plot_branch_selection(adata)
plt.show()

In [ ]:
palantir.plot.plot_trajectory(adata, terminal_cell1)

In [ ]:
palantir.plot.plot_trajectory(adata, terminal_cell2)

In [ ]:
gene_trends = palantir.presults.compute_gene_trends(adata, expression_key="MAGIC_imputed_data")

In [ ]:
hv_genes = adata.var_names[:500]
communities = palantir.presults.cluster_gene_trends(adata, terminal_cell1, hv_genes)

In [ ]:
communities.head()

In [ ]:
palantir.plot.plot_gene_trend_heatmaps(adata, hv_genes[:20])
plt.show()

In [ ]:
palantir.plot.plot_gene_trend_clusters(adata, terminal_cell1)
plt.show()



In [ ]:
branch1_adata = adata[masks[:,0]]

In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d

N = 100 # Number of genes
n_bins = 50 # Number of bins
smooth_window = 1

# Select top N highly variable genes by normalized dispersion
top_genes = (
    branch1_adata.var[branch1_adata.var["highly_variable"]]
    .nlargest(N, "dispersions_norm")
    .index.tolist()
)

# Extract expression matrix
X = branch1_adata[:, top_genes].X
if hasattr(X, "toarray"):
    X = X.toarray()
X = np.array(X, dtype=float)

# Sort cells by pseudotime (drop NaN cells like the root)
pt = branch1_adata.obs["dpt_pseudotime"].values
valid = np.isfinite(pt)
X, pt = X[valid], pt[valid]
order = np.argsort(pt)
X_sorted, pt_sorted = X[order], pt[order]

# Bin cells along pseudotime and average
bins = np.linspace(pt_sorted.min(), pt_sorted.max(), n_bins + 1)
bin_idx = np.clip(np.digitize(pt_sorted, bins) - 1, 0, n_bins - 1)
X_binned = np.array([X_sorted[bin_idx == b].mean(axis=0) if (bin_idx == b).any()
                     else np.zeros(len(top_genes)) for b in range(n_bins)])

# Smooth and z-score each gene
X_binned = uniform_filter1d(X_binned, size=smooth_window, axis=0)
std = X_binned.std(axis=0)
std[std == 0] = 1
X_z = (X_binned - X_binned.mean(axis=0)) / std

# Order genes by peak expression bin
gene_order = np.argsort(np.argmax(X_z, axis=0))
X_z = X_z[:, gene_order]
genes_ordered = [top_genes[i] for i in gene_order]

# Plot
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(X_z.T, aspect="auto", cmap="RdYlBu_r", vmin=-2, vmax=2)
ax.set_yticks(range(len(genes_ordered)))
ax.set_yticklabels(genes_ordered, fontsize=7)
ax.set_xlabel("Pseudotime →")
ax.set_title(f"Top {N} HVGs Along Pseudotime")
xtick_pos = np.linspace(0, n_bins - 1, 6, dtype=int)
ax.set_xticks(xtick_pos)
ax.set_xticklabels([f"{pt_sorted.min() + (pt_sorted.max() - pt_sorted.min()) * i / 5:.2f}"
                    for i in range(6)], fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.6, label="Z-score")
plt.tight_layout()
plt.show()

In [ ]:
branch2_adata = adata[masks[:,1]]

In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d

N = 100 # Number of genes
n_bins = 50 # Number of bins
smooth_window = 1

# Select top N highly variable genes by normalized dispersion
top_genes = (
    branch2_adata.var[branch2_adata.var["highly_variable"]]
    .nlargest(N, "dispersions_norm")
    .index.tolist()
)

# Extract expression matrix
X = branch2_adata[:, top_genes].X
if hasattr(X, "toarray"):
    X = X.toarray()
X = np.array(X, dtype=float)

# Sort cells by pseudotime (drop NaN cells like the root)
pt = branch2_adata.obs["dpt_pseudotime"].values
valid = np.isfinite(pt)
X, pt = X[valid], pt[valid]
order = np.argsort(pt)
X_sorted, pt_sorted = X[order], pt[order]

# Bin cells along pseudotime and average
bins = np.linspace(pt_sorted.min(), pt_sorted.max(), n_bins + 1)
bin_idx = np.clip(np.digitize(pt_sorted, bins) - 1, 0, n_bins - 1)
X_binned = np.array([X_sorted[bin_idx == b].mean(axis=0) if (bin_idx == b).any()
                     else np.zeros(len(top_genes)) for b in range(n_bins)])

# Smooth and z-score each gene
X_binned = uniform_filter1d(X_binned, size=smooth_window, axis=0)
std = X_binned.std(axis=0)
std[std == 0] = 1
X_z = (X_binned - X_binned.mean(axis=0)) / std

# Order genes by peak expression bin
gene_order = np.argsort(np.argmax(X_z, axis=0))
X_z = X_z[:, gene_order]
genes_ordered = [top_genes[i] for i in gene_order]

# Plot
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(X_z.T, aspect="auto", cmap="RdYlBu_r", vmin=-2, vmax=2)
ax.set_yticks(range(len(genes_ordered)))
ax.set_yticklabels(genes_ordered, fontsize=7)
ax.set_xlabel("Pseudotime →")
ax.set_title(f"Top {N} HVGs Along Pseudotime")
xtick_pos = np.linspace(0, n_bins - 1, 6, dtype=int)
ax.set_xticks(xtick_pos)
ax.set_xticklabels([f"{pt_sorted.min() + (pt_sorted.max() - pt_sorted.min()) * i / 5:.2f}"
                    for i in range(6)], fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.6, label="Z-score")
plt.tight_layout()
plt.show()